# Module 02: Pandas for Machine Learning
## Notebook 05: Combining Datasets and Time Series Manipulation

Machine learning models rarely consume a single isolated table. Feature engineering requires joining user demographics with transactional history, product catalogs, and temporal sensor streams. Furthermore, time-dependent data requires rigorous handling to prevent lookahead data leakage.

---

### Learning Objectives
By the end of this notebook, you will be able to:
1. Merge relational tables using inner, left, right, and outer joins (`pd.merge`).
2. Concatenate datasets along rows and columns (`pd.concat`).
3. Parse, index, and extract temporal components from `pd.to_datetime`.
4. Resample time series and engineer rolling window statistics (`rolling`).
5. Create historical lag features to prevent lookahead leakage in forecasting.
6. **Advanced:** Perform asynchronous nearest-timestamp matching (`pd.merge_asof`) and compute Exponential Moving Averages (`.ewm()`).

In [1]:
import pandas as pd
import numpy as np

# Create two relational tables: Customers and Orders
customers = pd.DataFrame({
    'Customer_ID': [1, 2, 3, 4],
    'Name': ['Alice', 'Bob', 'Charlie', 'David'],
    'Tier': ['Gold', 'Silver', 'Gold', 'Bronze']
})

orders = pd.DataFrame({
    'Order_ID': [501, 502, 503, 504, 505],
    'Customer_ID': [1, 2, 1, 5, 2],
    'Amount': [250.0, 120.0, 85.0, 310.0, 45.0]
})

print("Customers Table:\n", customers)
print("\nOrders Table:\n", orders)

Customers Table:
    Customer_ID     Name    Tier
0            1    Alice    Gold
1            2      Bob  Silver
2            3  Charlie    Gold
3            4    David  Bronze

Orders Table:
    Order_ID  Customer_ID  Amount
0       501            1   250.0
1       502            2   120.0
2       503            1    85.0
3       504            5   310.0
4       505            2    45.0


---
### 1. Merging Relational Tables: `pd.merge()`

Pandas offers database-style joins:
- **Inner Join (`how='inner'`):** Only records with matching keys in both tables.
- **Left Join (`how='left'`):** All records from the left table; missing keys filled with `NaN`.
- **Right Join (`how='right'`):** All records from the right table.
- **Outer Join (`how='outer'`):** All records from both tables.

In [2]:
# Left join: Preserve all customers even if they have zero orders
df_left = pd.merge(customers, orders, on='Customer_ID', how='left')
print("Left Join (All customers preserved):\n", df_left)

# Inner join: Only retain customers who placed an order
df_inner = pd.merge(customers, orders, on='Customer_ID', how='inner')
print("\nInner Join (Only matching customers):\n", df_inner)

Left Join (All customers preserved):
    Customer_ID     Name    Tier  Order_ID  Amount
0            1    Alice    Gold     501.0   250.0
1            1    Alice    Gold     503.0    85.0
2            2      Bob  Silver     502.0   120.0
3            2      Bob  Silver     505.0    45.0
4            3  Charlie    Gold       NaN     NaN
5            4    David  Bronze       NaN     NaN

Inner Join (Only matching customers):
    Customer_ID   Name    Tier  Order_ID  Amount
0            1  Alice    Gold       501   250.0
1            1  Alice    Gold       503    85.0
2            2    Bob  Silver       502   120.0
3            2    Bob  Silver       505    45.0


---
### 2. Concatenation: `pd.concat()`

`pd.concat()` stacks DataFrames together:
- `axis=0`: Stacks vertically (e.g., appending new monthly batches of observations).
- `axis=1`: Stacks horizontally (e.g., joining newly calculated feature matrices).

In [3]:
batch_jan = pd.DataFrame({'Sales': [100, 150], 'Units': [10, 15]}, index=['Day1', 'Day2'])
batch_feb = pd.DataFrame({'Sales': [200, 180], 'Units': [20, 18]}, index=['Day3', 'Day4'])

# Vertical concatenation
combined_batches = pd.concat([batch_jan, batch_feb], axis=0)
print("Vertically Concatenated Batches:\n", combined_batches)

Vertically Concatenated Batches:
       Sales  Units
Day1    100     10
Day2    150     15
Day3    200     20
Day4    180     18


---
### 3. DateTime Processing & Feature Extraction

Machine learning algorithms cannot directly ingest raw timestamp strings (e.g., `"2025-04-12 14:30:00"`).
Converting to `pd.to_datetime` unlocks the `.dt` accessor for extracting seasonal and cyclical features:

In [4]:
# Create time series DataFrame
date_rng = pd.date_range(start='2025-01-01', periods=8, freq='D')
ts_df = pd.DataFrame({
    'Timestamp': date_rng,
    'Revenue': [1200, 1350, 950, 1100, 1800, 2100, 1400, 1550]
})

# Feature extraction via .dt accessor
ts_df['Day_Of_Week'] = ts_df['Timestamp'].dt.dayofweek
ts_df['Day_Name'] = ts_df['Timestamp'].dt.day_name()
ts_df['Is_Weekend'] = ts_df['Timestamp'].dt.dayofweek.isin([5, 6]).astype(int)

print("Engineered Datetime Features:")
print(ts_df)

Engineered Datetime Features:
   Timestamp  Revenue  Day_Of_Week   Day_Name  Is_Weekend
0 2025-01-01     1200            2  Wednesday           0
1 2025-01-02     1350            3   Thursday           0
2 2025-01-03      950            4     Friday           0
3 2025-01-04     1100            5   Saturday           1
4 2025-01-05     1800            6     Sunday           1
5 2025-01-06     2100            0     Monday           0
6 2025-01-07     1400            1    Tuesday           0
7 2025-01-08     1550            2  Wednesday           0


---
### 4. Time Series Resampling & Rolling Statistics

In temporal machine learning (predicting demand, energy consumption, or financial asset prices):
- **Resampling (`.resample()`):** Changes frequency (e.g. hourly $\to$ daily $\to$ weekly) with aggregations.
- **Rolling Windows (`.rolling()`):** Computes moving averages, rolling volatilities, and smoothed trends over a lookback window $W$.
- **Lag Features (`.shift()`):** Shifts values forward in time so that time step $t$ only has access to $t-1, t-2$, preventing lookahead leakage.

In [5]:
# 3-day rolling mean (moving average)
ts_df['Rolling_Mean_3D'] = ts_df['Revenue'].rolling(window=3).mean().round(2)

# 1-day lag feature: Previous day's revenue
ts_df['Lag_1_Revenue'] = ts_df['Revenue'].shift(1)

# Day-over-day growth rate
ts_df['DoD_Growth'] = ((ts_df['Revenue'] - ts_df['Lag_1_Revenue']) / ts_df['Lag_1_Revenue']).round(3)

print("Temporal Features with Rolling Statistics and Lags:")
print(ts_df[['Timestamp', 'Revenue', 'Rolling_Mean_3D', 'Lag_1_Revenue', 'DoD_Growth']])

Temporal Features with Rolling Statistics and Lags:
   Timestamp  Revenue  Rolling_Mean_3D  Lag_1_Revenue  DoD_Growth
0 2025-01-01     1200              NaN            NaN         NaN
1 2025-01-02     1350              NaN         1200.0       0.125
2 2025-01-03      950          1166.67         1350.0      -0.296
3 2025-01-04     1100          1133.33          950.0       0.158
4 2025-01-05     1800          1283.33         1100.0       0.636
5 2025-01-06     2100          1666.67         1800.0       0.167
6 2025-01-07     1400          1766.67         2100.0      -0.333
7 2025-01-08     1550          1683.33         1400.0       0.107


---
### 5. Advanced Complex Usage: As-Of Merges (`pd.merge_asof`) and Exponential Moving Averages (`.ewm()`)

In real-world data science, timestamps across systems never align exactly:
- **Asynchronous Telemetry:** Trade orders occur at 10:00:00.125, while market quotes update at 10:00:00.118.
- An exact inner join yields 0 matches!
- **`pd.merge_asof`** matches on the nearest key rather than exact equality. With `direction='backward'`, it strictly matches the most recent past event, ensuring **zero future lookahead leakage**.

Additionally, simple rolling averages treat observations 30 days ago with equal weight to yesterday. **Exponentially Weighted Moving Averages (`.ewm()`)** assign exponentially decaying weights to older observations.

In [6]:
# 1. As-Of Merge Demonstration
# Trades executed asynchronously
trades = pd.DataFrame({
    'trade_time': pd.to_datetime(['2025-06-01 09:30:01', '2025-06-01 09:30:05', '2025-06-01 09:30:12']),
    'ticker': ['AAPL', 'AAPL', 'AAPL'],
    'shares': [100, 250, 50]
})

# Quotes recorded by market data feed at irregular intervals
quotes = pd.DataFrame({
    'quote_time': pd.to_datetime(['2025-06-01 09:30:00', '2025-06-01 09:30:03', '2025-06-01 09:30:08', '2025-06-01 09:30:15']),
    'ticker': ['AAPL', 'AAPL', 'AAPL', 'AAPL'],
    'bid': [182.10, 182.25, 182.20, 182.40],
    'ask': [182.15, 182.30, 182.25, 182.45]
})

# Perform backward as-of join within 5-second tolerance
trades_with_quotes = pd.merge_asof(
    trades,
    quotes,
    left_on='trade_time',
    right_on='quote_time',
    by='ticker',
    direction='backward',
    tolerance=pd.Timedelta('5s')
)

print("As-Of Joined Trades and Nearest Preceding Quotes:")
print(trades_with_quotes[['trade_time', 'ticker', 'shares', 'quote_time', 'bid', 'ask']])

# 2. Exponential Moving Average (.ewm) for trend tracking
simulated_prices = pd.Series([100, 102, 101, 105, 107, 115, 114, 118, 122, 120])
ema_span3 = simulated_prices.ewm(span=3, adjust=False).mean()
sma_win3 = simulated_prices.rolling(window=3).mean()

comparison_df = pd.DataFrame({
    'Price': simulated_prices,
    'SMA_3': sma_win3.round(2),
    'EMA_Span3': ema_span3.round(2)
})
print("\nSimple vs Exponential Moving Average:")
print(comparison_df)

As-Of Joined Trades and Nearest Preceding Quotes:
           trade_time ticker  shares          quote_time     bid     ask
0 2025-06-01 09:30:01   AAPL     100 2025-06-01 09:30:00  182.10  182.15
1 2025-06-01 09:30:05   AAPL     250 2025-06-01 09:30:03  182.25  182.30
2 2025-06-01 09:30:12   AAPL      50 2025-06-01 09:30:08  182.20  182.25

Simple vs Exponential Moving Average:
   Price   SMA_3  EMA_Span3
0    100     NaN     100.00
1    102     NaN     101.00
2    101  101.00     101.00
3    105  102.67     103.00
4    107  104.33     105.00
5    115  109.00     110.00
6    114  112.00     112.00
7    118  115.67     115.00
8    122  118.00     118.50
9    120  120.00     119.25


### Summary & Next Steps
In this notebook, you mastered:
- Relational joining strategies (`left`, `inner`, `right`, `outer`) via `pd.merge`.
- Concatenating datasets across row and column axes.
- Datetime parsing and temporal component extraction via `.dt`.
- Rolling windows, moving statistics, and shift-based lag features.
- Asynchronous nearest joins with `pd.merge_asof` and Exponentially Weighted Moving Averages (`.ewm`).

**Next Notebook:** `06_feature_engineering_with_pandas.ipynb` — One-hot encoding, binning, IQR outlier filtering, and out-of-fold target encoding.